# Phase 13.3 — Search API Serving

Walk through the HTTP search lifecycle: contract validation → serving service → existing production retrieval/ranking → API-safe JSON.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from fastapi.testclient import TestClient

from productiq.api import create_app
from productiq.serving import PLANNED_ROUTE_SEARCH, SearchApiRequest
from productiq.serving.search_service import ProductionSearchServingService
from tests.retrieval.test_production_ranking_integration import _ranked_pipeline

## 1. Build an in-memory production pipeline (no PostgreSQL)

The same `ProductionRetrievalPipeline.retrieve_ranked` path used in Phase 10.5 tests backs the HTTP service.

In [ ]:
pipeline, rrf, catalog = _ranked_pipeline(ranking=("P1", "P2", "P3"), pool_top_k=10)
search_service = ProductionSearchServingService(pipeline)
app = create_app(search_service=search_service)

## 2. POST /api/v1/search

The route stays thin; intelligence lives in the serving service and existing pipeline.

In [ ]:
with TestClient(app) as client:
    response = client.post(
        PLANNED_ROUTE_SEARCH,
        json=SearchApiRequest(query="nike shoes", top_k=2).model_dump(mode="json"),
    )
response.status_code, response.json()

## 3. Empty results are success

Hard filters that match nothing return `returned_count=0` without raising.

In [ ]:
from productiq.representation.query_contract import QueryFilterConstraints

body = {
    "query": "shoes",
    "top_k": 5,
    "filters": {"brand_normalized": "brand_that_does_not_exist"},
}
with TestClient(app) as client:
    empty = client.post(PLANNED_ROUTE_SEARCH, json=body)
empty.json()